In [2]:
import cv2
import numpy as np

In [ ]:
# 1a: Cargar una imagen y mostrar el valor RGB del píxel bajo el cursor

ruta_imagen = "Imagen1.jpg"
img = cv2.imread(ruta_imagen)

if img is None:
    raise FileNotFoundError(f"No se pudo cargar la imagen en: {ruta_imagen}")

# --- Escalar la imagen para que quepa en pantalla sin recortarse ---
ANCHO_MAX, ALTO_MAX = 1280, 720
alto_original, ancho_original = img.shape[:2]

escala = min(ANCHO_MAX / ancho_original, ALTO_MAX / alto_original, 1.0)
ancho_mostrado = int(ancho_original * escala)
alto_mostrado = int(alto_original * escala)

img_mostrada = cv2.resize(img, (ancho_mostrado, alto_mostrado), interpolation=cv2.INTER_AREA)

ventana = "Practica 1a - Valor RGB bajo el cursor"
cv2.namedWindow(ventana)


def mostrar_rgb(event, x, y, flags, param):
    if event == cv2.EVENT_MOUSEMOVE:
        # Convertimos la posición del ratón (sobre la imagen escalada)
        # a coordenadas de la imagen original, para leer el píxel real
        x_original = min(int(x / escala), ancho_original - 1)
        y_original = min(int(y / escala), alto_original - 1)

        # OpenCV almacena los píxeles en formato BGR
        b, g, r = img[y_original, x_original]
        texto = f"RGB: ({r}, {g}, {b})  Pos: ({x_original}, {y_original})"

        vista = img_mostrada.copy()
        cv2.putText(vista, texto, (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv2.imshow(ventana, vista)


cv2.setMouseCallback(ventana, mostrar_rgb)
cv2.imshow(ventana, img_mostrada)

print("Mueve el ratón sobre la imagen para ver el valor RGB del píxel.")
print("Pulsa 'q' o ESC para salir.")

while True:
    key = cv2.waitKey(20) & 0xFF
    if key == ord('q') or key == 27:
        break

cv2.destroyAllWindows()

Mueve el ratón sobre la imagen para ver el valor RGB del píxel.
Pulsa 'q' o ESC para salir.


In [ ]:
import cv2
import numpy as np

# 1b + Optativo: Dibujo avanzado de figuras primitivas, rellenas, y grabación paso a paso del vídeo
ANCHO, ALTO = 800, 600
lienzo = np.ones((ALTO, ANCHO, 3), dtype=np.uint8) * 255

ventana = "Practica 1b"
cv2.namedWindow(ventana)

def nada(x):
    pass

# Trackbars de control en la interfaz
cv2.createTrackbar("R", ventana, 0, 255, nada)
cv2.createTrackbar("G", ventana, 0, 255, nada)
cv2.createTrackbar("B", ventana, 255, 255, nada)
cv2.createTrackbar("Grosor", ventana, 2, 30, nada)
cv2.createTrackbar("Relleno", ventana, 0, 1, nada)

# Configuración del archivo de vídeo
FPS = 30.0
NOMBRE_VIDEO = "dibujo_paso_a_paso.mp4"
DURACION_TRAZO_SEG = 1.0  # Tiempo que se muestra cada paso en el vídeo
FRAMES_POR_PASO = int(FPS * DURACION_TRAZO_SEG)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
grabador = cv2.VideoWriter(NOMBRE_VIDEO, fourcc, FPS, (ANCHO, ALTO))

# Modos disponibles: 'linea', 'rectangulo', 'circulo', 'elipse', 'poligono'
modo = "linea"
dibujando = False
pt_inicio = (0, 0)
puntos_poligono = []
lienzo_temporal = lienzo.copy()


def color_actual():
    r = cv2.getTrackbarPos("R", ventana)
    g = cv2.getTrackbarPos("G", ventana)
    b = cv2.getTrackbarPos("B", ventana)
    return (b, g, r)


def grosor_actual():
    relleno = cv2.getTrackbarPos("Relleno", ventana)
    if relleno == 1:
        return -1
    return max(1, cv2.getTrackbarPos("Grosor", ventana))


def registrar_paso(imagen, repeticiones=FRAMES_POR_PASO):
    """Escribe la imagen confirmada repetidas veces para dar duración al fotograma en el vídeo."""
    if grabador.isOpened():
        for _ in range(repeticiones):
            grabador.write(imagen)


def dibujar_primitiva(destino, x, y):
    color = color_actual()
    grosor = grosor_actual()

    if modo == "linea":
        grosor_linea = max(1, cv2.getTrackbarPos("Grosor", ventana))
        cv2.line(destino, pt_inicio, (x, y), color, grosor_linea)

    elif modo == "rectangulo":
        cv2.rectangle(destino, pt_inicio, (x, y), color, grosor)

    elif modo == "circulo":
        radio = int(np.hypot(x - pt_inicio[0], y - pt_inicio[1]))
        cv2.circle(destino, pt_inicio, radio, color, grosor)

    elif modo == "elipse":
        centro = pt_inicio
        eje_x = max(1, abs(x - pt_inicio[0]))
        eje_y = max(1, abs(y - pt_inicio[1]))
        cv2.ellipse(destino, centro, (eje_x, eje_y), 0, 0, 360, color, grosor)


def finalizar_poligono():
    global puntos_poligono, lienzo
    if len(puntos_poligono) >= 2:
        color = color_actual()
        grosor = grosor_actual()
        pts = np.array(puntos_poligono, np.int32).reshape((-1, 1, 2))

        if grosor == -1:
            cv2.fillPoly(lienzo, [pts], color)
        else:
            cv2.polylines(lienzo, [pts], isClosed=True, color=color, thickness=grosor)

        # Graba el paso una vez el polígono quedó sellado en el lienzo
        registrar_paso(lienzo)

    puntos_poligono.clear()
    cv2.imshow(ventana, lienzo)


def evento_raton(event, x, y, flags, param):
    global dibujando, pt_inicio, lienzo, lienzo_temporal, puntos_poligono

    if modo == "poligono":
        if event == cv2.EVENT_LBUTTONDOWN:
            puntos_poligono.append((x, y))
            vista = lienzo.copy()
            for pt in puntos_poligono:
                cv2.circle(vista, pt, 3, (0, 0, 255), -1)
            if len(puntos_poligono) > 1:
                pts = np.array(puntos_poligono, np.int32).reshape((-1, 1, 2))
                cv2.polylines(vista, [pts], isClosed=False, color=color_actual(), thickness=max(1, cv2.getTrackbarPos("Grosor", ventana)))
            cv2.imshow(ventana, vista)

        elif event == cv2.EVENT_MOUSEMOVE and len(puntos_poligono) > 0:
            vista = lienzo.copy()
            pts = np.array(puntos_poligono, np.int32).reshape((-1, 1, 2))
            grosor_linea = max(1, cv2.getTrackbarPos("Grosor", ventana))
            cv2.polylines(vista, [pts], isClosed=False, color=color_actual(), thickness=grosor_linea)
            cv2.line(vista, puntos_poligono[-1], (x, y), color_actual(), 1)
            cv2.imshow(ventana, vista)

        elif event == cv2.EVENT_RBUTTONDOWN or event == cv2.EVENT_LBUTTONDBLCLK:
            finalizar_poligono()

    else:
        if event == cv2.EVENT_LBUTTONDOWN:
            dibujando = True
            pt_inicio = (x, y)
            lienzo_temporal = lienzo.copy()

        elif event == cv2.EVENT_MOUSEMOVE and dibujando:
            vista = lienzo_temporal.copy()
            dibujar_primitiva(vista, x, y)
            cv2.imshow(ventana, vista)

        elif event == cv2.EVENT_LBUTTONUP:
            dibujando = False
            dibujar_primitiva(lienzo, x, y)
            cv2.imshow(ventana, lienzo)
            # Graba el nuevo estado confirmado del lienzo
            registrar_paso(lienzo)


cv2.setMouseCallback(ventana, evento_raton)
cv2.imshow(ventana, lienzo)

registrar_paso(lienzo)

print("Controles:")
print("  l -> modo línea")
print("  r -> modo rectángulo")
print("  c -> modo círculo")
print("  e -> modo elipse")
print("  p -> modo polígono (Clic izq: vértice | Clic der / Doble clic: confirmar)")
print("  f -> alternar Relleno")
print("  n -> limpiar el lienzo (se registra como nuevo paso)")
print("  q / ESC -> finalizar y guardar vídeo")

while True:
    key = cv2.waitKey(20) & 0xFF
    if key == ord('q') or key == 27:
        break
    elif key == ord('l'):
        modo = "linea"
        puntos_poligono.clear()
        print("Modo: línea")
    elif key == ord('r'):
        modo = "rectangulo"
        puntos_poligono.clear()
        print("Modo: rectángulo")
    elif key == ord('c'):
        modo = "circulo"
        puntos_poligono.clear()
        print("Modo: círculo")
    elif key == ord('e'):
        modo = "elipse"
        puntos_poligono.clear()
        print("Modo: elipse")
    elif key == ord('p'):
        modo = "poligono"
        puntos_poligono.clear()
        print("Modo: polígono")
    elif key == ord('f'):
        actual = cv2.getTrackbarPos("Relleno", ventana)
        nuevo = 0 if actual == 1 else 1
        cv2.setTrackbarPos("Relleno", ventana, nuevo)
        print(f"Relleno: {'ACTIVADO' if nuevo == 1 else 'DESACTIVADO'}")
    elif key == ord('n'):
        lienzo[:] = 255
        lienzo_temporal = lienzo.copy()
        puntos_poligono.clear()
        cv2.imshow(ventana, lienzo)
        registrar_paso(lienzo)
        print("Lienzo reiniciado (registrado en el vídeo)")

# Congelar el dibujo terminado al final durante 2 segundos antes de cerrar
registrar_paso(lienzo, repeticiones=int(FPS * 2.0))

# Liberar el archivo de vídeo
grabador.release()
cv2.destroyAllWindows()
print(f"Vídeo de la secuencia guardado exitosamente en: '{NOMBRE_VIDEO}'.")

Controles:
  l -> modo línea
  r -> modo rectángulo
  c -> modo círculo
  e -> modo elipse
  p -> modo polígono (Clic izq: vértice | Clic der / Doble clic: confirmar)
  f -> alternar Relleno
  n -> limpiar el lienzo (se registra como nuevo paso)
  q / ESC -> finalizar y guardar vídeo
Relleno: ACTIVADO
Relleno: DESACTIVADO
Modo: círculo
Relleno: ACTIVADO
Vídeo de la secuencia guardado exitosamente en: 'dibujo_paso_a_paso.mp4'.
